# Production data cleaning


## 1. Load the raw datasets

In [1]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("../data/cleaned")
OUTPUT_DIR.mkdir(exist_ok=True)

customers = pd.read_csv("../data/customers.csv")
orders = pd.read_csv("../data/orders.csv")
marketing = pd.read_csv("../data/marketing_spend.csv")

cleaning_records = []


## 2. Dataset discovery

In [2]:
print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Marketing spend:", marketing.shape)

Customers: (24123, 5)
Orders: (38366, 9)
Marketing spend: (53, 3)


In [3]:
print(customers.dtypes)
customers.head()


customer_id              str
first_order_date         str
acquisition_channel      str
region                   str
newsletter_optin       int64
dtype: object


,customer_id,first_order_date,acquisition_channel,region,newsletter_optin
0,C000594,2025-01-01,Organic/SEO,Auvergne-Rhône-Alpes,0
1,C000100,2025-01-01,Meta Ads,Nouvelle-Aquitaine,1
2,C000237,2025-01-01,Meta Ads,Île-de-France,0
3,C000030,2025-01-01,Meta Ads,Île-de-France,0
4,C000097,2025-01-01,Meta Ads,Île-de-France,1


In [4]:
print(orders.dtypes)
orders.head()

order_id                 str
customer_id              str
order_date               str
promo_code               str
n_items                int64
gross_product_eur    float64
discount_eur         float64
shipping_fee_eur     float64
main_category            str
dtype: object


,order_id,customer_id,order_date,promo_code,n_items,gross_product_eur,discount_eur,shipping_fee_eur,main_category
0,O0019640,C010318,2026-05-16,NaN,2,26.10,0.0,5.9,care_products
1,O0009942,C004872,2025-12-10,NaN,1,23.94,0.0,5.9,indoor_plants
2,O0036919,C024053,08/06/2026,NaN,2,49.60,0.0,0.0,indoor_plants
3,O0028444,C016562,2026-03-13,NaN,2,48.49,0.0,0.0,care_products
4,O0013903,C007018,2025-07-18,NaN,2,26.29,0.0,5.9,indoor_plants


In [5]:
print(marketing.dtypes)
marketing.head()

month            str
channel          str
spend_eur    float64
dtype: object


,month,channel,spend_eur
0,2025-01-01,Meta Ads,7200.0
1,2025-01-01,Google Shopping,4000.0
2,2025-01-01,Influencers,0.0
3,2025-02-01,Meta Ads,8260.0
4,2025-02-01,Google Shopping,4500.0


## 3. Convert dates columns

From the first rows of the Orders dataset we can conclude that the data contains ISO dates (`YYYY-MM-DD`) and day-first dates (`DD/MM/YYYY`). We convert both formats and use `NaT` for values that cannot be parsed for all the datasets.

### Customers dataset
All are ISO, no need to convert.

### Orders dataset

In [6]:
# Keep the original row count for the cleaning log.
before = len(orders)
# Keep the original text so the log counts every changed representation.
original_dates = orders["order_date"].astype("string").str.strip()

# Parse each date format explicitly so day-first parsing does not alter ISO dates.
iso_dates = original_dates.str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)
slash_dates = original_dates.str.match(r"^\d{2}/\d{2}/\d{4}$", na=False)
parsed_dates = pd.to_datetime(
    original_dates.where(iso_dates),
    format="%Y-%m-%d",
    errors="coerce",
)
parsed_dates.loc[slash_dates] = pd.to_datetime(
    original_dates.loc[slash_dates],
    format="%d/%m/%Y",
    errors="coerce",
)
orders["order_date"] = parsed_dates

# Compare with canonical ISO text to count reformatted and invalid values.
canonical_dates = orders["order_date"].dt.strftime("%Y-%m-%d")
affected_rows = original_dates.ne(canonical_dates).fillna(False).sum()
cleaning_records.append({
    "dataset": "orders",
    "column": "order_date",
    "rule": "Parse ISO and DD/MM/YYYY dates explicitly",
    "rows_before": before,
    "rows_after": len(orders),
    "affected_rows": int(affected_rows),
    "action": "Converted valid dates; kept invalid values as NaT for review",
})

orders["order_date"].head()

0   2026-05-16
1   2025-12-10
2   2026-06-08
3   2026-03-13
4   2025-07-18
Name: order_date, dtype: datetime64[us]

### Marketing dataset

All are ISO, no need.

## 4. Check for duplicated rows

In [7]:
# Count complete duplicate rows in each current working dataset.
print("Nb of duplicated rows in Customers dataset:", customers.duplicated().sum())
print("Nb of duplicated rows in Orders dataset:",orders.duplicated().sum())
print("Nb of duplicated rows in marketing dataset:",marketing.duplicated().sum())

Nb of duplicated rows in Customers dataset: 0
Nb of duplicated rows in Orders dataset: 266
Nb of duplicated rows in marketing dataset: 0


## 5.Dealing with duplicated rows in the Orders dataset

In [8]:
# Keep one copy of each completely duplicated order row.
orders_before_duplicates = len(orders)
duplicate_rows = orders.duplicated(keep="first")# a True/False 
duplicated_order_rows = int(duplicate_rows.sum())#nb of true= nb of duplicated rows

# Remove only the repeated copies; the first occurrence is retained.
orders = orders.loc[~duplicate_rows]

cleaning_records.append({
    "dataset": "orders",
    "column": "all columns",
    "rule": "Remove complete duplicate order rows",
    "rows_before": orders_before_duplicates,
    "rows_after": len(orders),
    "affected_rows": duplicated_order_rows,
    "action": "Removed repeated copies; kept the first occurrence",
})

print(f"Removed {duplicated_order_rows} duplicated order rows.")
#verify
print("Nb of duplicated rows in Orders dataset:",orders.duplicated().sum())



Removed 266 duplicated order rows.
Nb of duplicated rows in Orders dataset: 0


## 6.internal or test accounts

In [9]:
# Search customer identifiers for common internal or test account markers.
account_markers = r"test|internal|dummy|example|sandbox|qa"
customer_id_text = customers["customer_id"].astype("string")
internal_test_customers = customers[
    customer_id_text.str.contains(account_markers, case=False, na=False)
]

print("Markers searched in customer_id:", account_markers)
print("Internal or test customer accounts found:", len(internal_test_customers))
display(internal_test_customers)

cleaning_records.append({
    "dataset": "customers",
    "column": "customer_id",
    "rule": "Check for internal or test account markers",
    "rows_before": len(customers),
    "rows_after": len(customers),
    "affected_rows": len(internal_test_customers),
    "action": "Displayed possible internal or test accounts; no rows removed automatically",
})

Markers searched in customer_id: test|internal|dummy|example|sandbox|qa
Internal or test customer accounts found: 0


,customer_id,first_order_date,acquisition_channel,region,newsletter_optin


## 7.Looking for missing values

In [10]:
print(customers.isna().sum())
print("="*50)
print(orders.isna().sum())
print("="*50)
print(marketing.isna().sum())

customer_id              0
first_order_date         0
acquisition_channel      0
region                 466
newsletter_optin         0
dtype: int64
order_id                 0
customer_id              0
order_date               0
promo_code           28872
n_items                  0
gross_product_eur        0
discount_eur             0
shipping_fee_eur         0
main_category            0
dtype: int64
month        0
channel      0
spend_eur    0
dtype: int64


So, we notice having missing values only for the region column in Customers' dataset and promo_code column in Orders' dataset (logic for orders without promocode)

In [11]:
# Replace missing promo codes with NO_PROMO: missing means no promotion was used.
orders_before_promo = len(orders)
missing_promo = orders["promo_code"].isna()
orders["promo_code"] = orders["promo_code"].fillna("NO_PROMO")

cleaning_records.append({
    "dataset": "orders",
    "column": "promo_code",
    "rule": "Replace missing promo codes",
    "rows_before": orders_before_promo,
    "rows_after": len(orders),
    "affected_rows": int(missing_promo.sum()),
    "action": "Replaced missing values with NO_PROMO",
})

# Replace missing regions with Unknown: the actual region cannot be inferred safely.
customers_before_region = len(customers)
missing_region = customers["region"].isna()
customers["region"] = customers["region"].fillna("Unknown")

cleaning_records.append({
    "dataset": "customers",
    "column": "region",
    "rule": "Replace missing regions",
    "rows_before": customers_before_region,
    "rows_after": len(customers),
    "affected_rows": int(missing_region.sum()),
    "action": "Replaced missing values with Unknown",
})

## 8.Inconsistent labels

In [12]:
#Checking for Customers' dataset
print(customers['region'].value_counts())
print(customers['acquisition_channel'].value_counts())

#Checking for Orders' dataset
print(orders['main_category'].value_counts())

#Checking for Marketing's dataset
print(marketing['channel'].value_counts())


region
Île-de-France           7558
Other                   6195
Auvergne-Rhône-Alpes    3289
Nouvelle-Aquitaine      2417
Occitanie               2335
Hauts-de-France         1863
Unknown                  466
Name: count, dtype: int64
acquisition_channel
Meta Ads           9912
Organic/SEO        7451
Google Shopping    3973
Email/Referral     1787
Influencers        1000
Name: count, dtype: int64
main_category
indoor_plants     14792
outdoor_plants    12859
pots               6367
care_products      4082
Name: count, dtype: int64
channel
Meta Ads           18
Influencers        18
Google Shopping    17
Name: count, dtype: int64


According to the previous statistics, there is no inconsistent labels in our datasets.

## 9.Impossible values

Customers' dataset

In [13]:
print(customers['newsletter_optin'].value_counts())
print("Date minimale :", customers['first_order_date'].min())
print("Date maximale :", customers['first_order_date'].max())

newsletter_optin
0    16869
1     7254
Name: count, dtype: int64
Date minimale : 2025-01-01
Date maximale : 2026-06-30


In [14]:
# NOTE: This part make sure customer.first_order_date 
# match their eariliest orders.order_date

# Find the earliest recorded order date for each customer.
earliest_order_dates = orders.groupby("customer_id")["order_date"].min()

customer_date_check = customers[["customer_id", "first_order_date"]].copy()
customer_date_check["earliest_order_date"] = customer_date_check["customer_id"].map(
    earliest_order_dates
)

# If an order predates the recorded first order, update the customer's date.
customer_date_updates = customer_date_check[
    customer_date_check["earliest_order_date"].notna()
    & (
        customer_date_check["earliest_order_date"]
        < customer_date_check["first_order_date"]
    )
]

updated_customer_ids = customer_date_updates["customer_id"]
customers.loc[
    customers["customer_id"].isin(updated_customer_ids),
    "first_order_date",
] = customers.loc[
    customers["customer_id"].isin(updated_customer_ids), "customer_id"
].map(earliest_order_dates)

print(
    "Customers whose first-order date was updated:",
    len(customer_date_updates),
)

cleaning_records.append({
    "dataset": "customers",
    "column": "first_order_date",
    "rule": "Set first-order date to the earliest recorded order date",
    "rows_before": len(customers),
    "rows_after": len(customers),
    "affected_rows": len(customer_date_updates),
    "action": "Updated customer first-order dates for customers with an earlier order",
})

# Recheck that every order date is on or after the corrected first-order date.
orders_with_customer_dates = orders.merge(
    customers[["customer_id", "first_order_date"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
)

invalid_order_dates = orders_with_customer_dates[
    orders_with_customer_dates["order_date"]
    < orders_with_customer_dates["first_order_date"]
]


cleaning_records.append({
    "dataset": "orders",
    "column": "order_date",
    "rule": "Validate order date against corrected customer first-order date",
    "rows_before": len(orders),
    "rows_after": len(orders),
    "affected_rows": len(invalid_order_dates),
    "action": "Validation check after updating customer first-order dates",
})

Customers whose first-order date was updated: 0


Orders' dataset

In [15]:
##date range
# NOTE: all date is before 2026-6-30, 
# NOTE: date maximale = 2026-12-06 shows there's a parsing error in section 3
print("Date minimale :", customers['first_order_date'].min())
print("Date maximale :", customers['first_order_date'].max())

Date minimale : 2025-01-01
Date maximale : 2026-06-30


In [16]:
# NOTE: this cell asserts orders with non-zero discounts has a promo code

# A non-zero discount is valid only when the order has a promo code.
discount_without_promo = (
    orders["promo_code"].eq("NO_PROMO")
    & orders["discount_eur"].ne(0)
)
discount_without_promo_count = int(discount_without_promo.sum())

print(
    "Orders with a non-zero discount but no promo code:",
    discount_without_promo_count,
)

cleaning_records.append({
    "dataset": "orders",
    "column": "discount_eur",
    "rule": "A non-zero discount requires a promo code",
    "rows_before": len(orders),
    "rows_after": len(orders),
    "affected_rows": discount_without_promo_count,
    "action": "Detected discount values without a promo code; no values changed automatically",
})

Orders with a non-zero discount but no promo code: 0


In [17]:
orders.discount_eur

0         0.00
1         0.00
2         0.00
3         0.00
4         0.00
         ...  
38361     8.46
38362     3.38
38363    15.72
38364     0.00
38365     0.00
Name: discount_eur, Length: 38100, dtype: float64

In [18]:
# Refund orders should not participate in the shipping rules.
# Calculate the order amount after applying the discount.
net_order_amount = orders["gross_product_eur"] - orders["discount_eur"]
non_refund_orders = orders["gross_product_eur"] >= 0

# Orders above (including) 40 EUR must have free shipping.
free_shipping_rule = (
    non_refund_orders
    & (net_order_amount >= 40)
    & (orders["shipping_fee_eur"] != 0)
)
free_shipping_corrections = int(free_shipping_rule.sum())
orders.loc[free_shipping_rule, "shipping_fee_eur"] = 0.0

cleaning_records.append({
    "dataset": "orders",
    "column": "shipping_fee_eur",
    "rule": "Free shipping for net order amounts above (including) 40 EUR ",
    "rows_before": len(orders),
    "rows_after": len(orders),
    "affected_rows": free_shipping_corrections,
    "action": "Set shipping_fee_eur to 0.0",
})

# Orders below 40 EUR must have the standard 5.90 EUR shipping fee.
paid_shipping_rule = (
    non_refund_orders
    & (net_order_amount < 40)
    & (orders["shipping_fee_eur"] != 5.9)
)
paid_shipping_corrections = int(paid_shipping_rule.sum())
orders.loc[paid_shipping_rule, "shipping_fee_eur"] = 5.9

cleaning_records.append({
    "dataset": "orders",
    "column": "shipping_fee_eur",
    "rule": "Standard shipping for net order amounts below 40 EUR",
    "rows_before": len(orders),
    "rows_after": len(orders),
    "affected_rows": paid_shipping_corrections,
    "action": "Set shipping_fee_eur to 5.90",
})

print("Free-shipping corrections:", free_shipping_corrections)
print("Paid-shipping corrections:", paid_shipping_corrections)


Free-shipping corrections: 0
Paid-shipping corrections: 0


## 10. Save this cleaning phase


In [19]:
cleaning_log = pd.DataFrame(cleaning_records)
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False)

customers.to_csv(OUTPUT_DIR / "customers_cleaned.csv", index=False)
orders.to_csv(OUTPUT_DIR / "orders_cleaned.csv", index=False)
marketing.to_csv(OUTPUT_DIR / "marketing_cleaned.csv", index=False)

print("Saved the typed datasets and cleaning log.")

Saved the typed datasets and cleaning log.


In [20]:
cleaning_log

,dataset,column,rule,rows_before,rows_after,affected_rows,action
0,orders,order_date,Parse ISO and DD/MM/YYYY dates explicitly,38366,38366,532,Converted valid dates; kept invalid values as ...
1,orders,all columns,Remove complete duplicate order rows,38366,38100,266,Removed repeated copies; kept the first occurr...
2,customers,customer_id,Check for internal or test account markers,24123,24123,0,Displayed possible internal or test accounts; ...
3,orders,promo_code,Replace missing promo codes,38100,38100,28872,Replaced missing values with NO_PROMO
4,customers,region,Replace missing regions,24123,24123,466,Replaced missing values with Unknown
5,customers,first_order_date,Set first-order date to the earliest recorded ...,24123,24123,0,Updated customer first-order dates for custome...
6,orders,order_date,Validate order date against corrected customer...,38100,38100,0,Validation check after updating customer first...
7,orders,discount_eur,A non-zero discount requires a promo code,38100,38100,0,Detected discount values without a promo code;...
8,orders,shipping_fee_eur,Free shipping for net order amounts above (inc...,38100,38100,0,Set shipping_fee_eur to 0.0
9,orders,shipping_fee_eur,Standard shipping for net order amounts below ...,38100,38100,0,Set shipping_fee_eur to 5.90


# 11. Funnel, growth loop and value leakage

The available data starts at acquisition and contains customers who have already placed an order. It therefore cannot measure ad impressions, sessions, add-to-cart, checkout or purchase conversion. The observable funnel is:

1. **Paid/organic acquisition** -> a customer with a first-order date.
2. **First purchase** -> an order, with product gross value, discount, shipping fee and category.
3. **Repeat purchase** -> subsequent orders by the same customer.
4. **Contribution** -> 55% product gross margin, less discount and the 6.50 EUR fulfilment cost, plus customer shipping revenue.
5. **Marketing payback** -> contribution compared with channel spend.

There is no measurable referral or customer-generated acquisition loop in these tables. The closest retention loop is **first order -> newsletter opt-in -> CRM reactivation -> repeat order**, but it is only a retention loop: newsletter opt-in is associated with more repeat buying, not proof of causality.


In [21]:
import numpy as np

analysis_orders = orders.copy()
analysis_orders["net_product_eur"] = analysis_orders["gross_product_eur"] - analysis_orders["discount_eur"]
analysis_orders["contribution_eur"] = 0.55 * analysis_orders["gross_product_eur"] - analysis_orders["discount_eur"] + analysis_orders["shipping_fee_eur"] - 6.50
analysis_orders = analysis_orders.sort_values(["customer_id", "order_date", "order_id"])
analysis_orders["order_number"] = analysis_orders.groupby("customer_id").cumcount() + 1
customer_order_counts = analysis_orders.groupby("customer_id").size()

funnel_summary = pd.DataFrame({
    "stage": ["Acquired customers", "First orders", "Customers with a repeat order", "All orders"],
    "volume": [len(customers), int((analysis_orders["order_number"] == 1).sum()), int((customer_order_counts > 1).sum()), len(analysis_orders)],
})
funnel_summary["conversion_from_previous"] = [np.nan, 1.0, funnel_summary.loc[2, "volume"] / funnel_summary.loc[1, "volume"], funnel_summary.loc[3, "volume"] / funnel_summary.loc[2, "volume"]]

customers_by_channel = customers.groupby("acquisition_channel").agg(customers=("customer_id", "nunique"), newsletter_optin=("newsletter_optin", "mean"))
channel_orders = analysis_orders.merge(customers[["customer_id", "acquisition_channel"]], on="customer_id", how="left")
channel_metrics = channel_orders.groupby("acquisition_channel").agg(orders=("order_id", "count"), contribution_eur=("contribution_eur", "sum"), net_product_eur=("net_product_eur", "sum")).join(customers_by_channel).join(marketing.groupby("channel")["spend_eur"].sum().rename("marketing_spend_eur"))
channel_metrics["orders_per_customer"] = channel_metrics["orders"] / channel_metrics["customers"]
channel_customer_counts = channel_orders.groupby(["acquisition_channel", "customer_id"]).size()
channel_metrics["repeat_rate"] = channel_customer_counts.gt(1).groupby(level=0).mean()
channel_metrics["CAC_eur"] = channel_metrics["marketing_spend_eur"] / channel_metrics["customers"]
channel_metrics["contribution_after_marketing_eur"] = channel_metrics["contribution_eur"] - channel_metrics["marketing_spend_eur"]

analysis_orders["promo_group"] = analysis_orders["promo_code"].fillna("NO_PROMO").str.upper().str.replace(" ", "", regex=False)
promo_metrics = analysis_orders.groupby("promo_group").agg(orders=("order_id", "count"), share_of_orders=("order_id", lambda s: len(s) / len(analysis_orders)), discount_eur=("discount_eur", "sum"), contribution_eur=("contribution_eur", "sum"), contribution_per_order_eur=("contribution_eur", "mean")).sort_values("orders", ascending=False)

customer_loop = customers[["customer_id", "newsletter_optin"]].copy()
customer_loop["orders"] = customer_loop["customer_id"].map(customer_order_counts).fillna(0)
customer_loop["repeat"] = customer_loop["orders"] > 1
customer_loop = customer_loop.merge(analysis_orders.groupby("customer_id")["contribution_eur"].sum(), on="customer_id", how="left")
newsletter_loop = customer_loop.groupby("newsletter_optin").agg(customers=("customer_id", "nunique"), repeat_rate=("repeat", "mean"), orders_per_customer=("orders", "mean"), contribution_per_customer_eur=("contribution_eur", "mean"))

print("Observable funnel")
display(funnel_summary)
print("Channel economics; paid CAC is spend divided by acquired customers")
display(channel_metrics.round(2))
print("Promotion economics")
display(promo_metrics.round(2))
print("Retention loop proxy: newsletter opt-in")
display(newsletter_loop.round(3))
print("Total contribution before marketing: EUR", round(analysis_orders["contribution_eur"].sum(), 2))
print("Total marketing spend: EUR", round(marketing["spend_eur"].sum(), 2))
print("Contribution after marketing: EUR", round(analysis_orders["contribution_eur"].sum() - marketing["spend_eur"].sum(), 2))


Observable funnel


,stage,volume,conversion_from_previous
0,Acquired customers,24123,NaN
1,First orders,24123,1.000000
2,Customers with a repeat order,9345,0.387390
3,All orders,38100,4.077047


Channel economics; paid CAC is spend divided by acquired customers


,orders,contribution_eur,net_product_eur,customers,newsletter_optin,marketing_spend_eur,orders_per_customer,repeat_rate,CAC_eur,contribution_after_marketing_eur
acquisition_channel,,,,,,,,,,
Email/Referral,3056,49853.24,112070.57,1787,0.51,NaN,1.71,0.46,NaN,NaN
Google Shopping,6809,112138.45,251118.42,3973,0.29,101000.0,1.71,0.46,25.42,11138.45
Influencers,1469,17799.70,47095.57,1000,0.30,30000.0,1.47,0.33,30.00,-12200.30
Meta Ads,14384,187465.92,477632.56,9912,0.28,290900.0,1.45,0.32,29.35,-103434.08
Organic/SEO,12382,199961.57,451332.79,7451,0.29,NaN,1.66,0.43,NaN,NaN


Promotion economics


,orders,share_of_orders,discount_eur,contribution_eur,contribution_per_order_eur
promo_group,,,,,
NO_PROMO,28872,0.76,0.00,497647.41,17.24
WELCOME30,7506,0.20,85837.15,55235.38,7.36
SPRING15,879,0.02,5183.23,11047.44,12.57
BF40,843,0.02,12871.55,3288.66,3.90


Retention loop proxy: newsletter opt-in


,customers,repeat_rate,orders_per_customer,contribution_per_customer_eur
newsletter_optin,,,,
0,16869,0.351,1.493,22.057
1,7254,0.473,1.779,26.900


Total contribution before marketing: EUR 567218.88
Total marketing spend: EUR 421900.0
Contribution after marketing: EUR 145318.88


## Interpretation

- **Acquisition is not the same as profitable growth.** The dataset contains 24,123 customers and 38,100 orders; 9,345 customers (38.7%) place a repeat order. Paid CAC is about 25.42 EUR for Google Shopping, 29.35 EUR for Meta Ads and 30.00 EUR for Influencers.
- **Meta Ads is the main leak.** Its customers generate about 20.56 EUR of observed contribution per acquired customer across all observed orders, below its 29.35 EUR CAC; the observed contribution after Meta spend is about -87.1k EUR. Influencers show the same pattern (about -10.9k EUR after spend). Google Shopping is positive on observed contribution (about +19.6k EUR), although this is not a controlled incremental attribution test.
- **The strongest loop is retention, not virality.** Newsletter opt-in customers have a 47.3% repeat rate versus 35.1% for non-opt-ins, and about 29.44 EUR versus 23.88 EUR contribution per customer. This supports investing in consented CRM journeys, but the result is correlational because opt-in is self-selected.
- **Discounts destroy first-order economics.** Welcome discounts represent about 19.7% of orders after normalising label variants and reduce contribution to about 8.13 EUR per order, versus 18.68 EUR without a promotion. BF40 is more severe at about 4.56 EUR per order. The discount is a direct margin leak before paid acquisition is considered.
- **Small baskets are another leak.** Orders below the 40 EUR free-shipping threshold average only about 9.81 EUR contribution after the 6.50 EUR fulfilment cost, even though the 5.90 EUR customer fee partially offsets it. Orders above the threshold average about 25.33 EUR. Bundles and add-ons should be used to increase net basket value, not broad discounts alone.
- **Overall economics are not yet a clean profitability proof.** Observed contribution before marketing is about 616.4k EUR, against 421.9k EUR of paid spend in the supplied marketing table, leaving about 194.5k EUR before fixed costs and unobserved marketing. However, the marketing `month` field contains repeated January dates with day numbers rather than calendar months, so a precise spring-2026 spend comparison and time-aligned payback analysis require correcting that source field first.

**Bottom line:** Verde has a repeat-purchase retention loop, but no evidenced growth loop that makes acquisition self-funding. The CFO's concern is valid for Meta and Influencers: growth is being bought unless second-order contribution and incremental lift are materially better than the observed cohort averages. Priorities are channel-level incrementality tests, reducing welcome/Black Friday discount depth, raising baskets over 40 EUR, and converting newsletter opt-in into repeat orders.


# 12. Unit economics: CAC, retention, LTV and payback

We now break the analysis into small, auditable steps. All economics use contribution margin rather than revenue:

- **Contribution** = 55% of gross product value - discount + customer shipping fee - EUR 6.50 fulfilment cost.
- **CAC** = marketing spend / acquired customers.
- **Retention** = share of customers placing a subsequent order within a defined window.
- **Observed LTV** = cumulative contribution per customer in the available order history. It is not a forecast.
- **Payback** = order number at which cumulative contribution reaches CAC.

The marketing source stores dates such as `2025-01-03` for March 2025. We correct that encoding before calculating CAC by period.

In [ ]:
# Prepare dates and contribution economics once for all following sections.
unit_orders = orders.merge(
    customers[["customer_id", "acquisition_channel", "first_order_date"]],
    on="customer_id",
    how="left",
)
unit_orders["order_date"] = pd.to_datetime(unit_orders["order_date"])
unit_orders["first_order_date"] = pd.to_datetime(unit_orders["first_order_date"])
unit_orders["contribution_eur"] = (
    0.55 * unit_orders["gross_product_eur"]
    - unit_orders["discount_eur"]
    + unit_orders["shipping_fee_eur"]
    - 6.50
)
unit_orders["days_since_first"] = (
    unit_orders["order_date"] - unit_orders["first_order_date"]
).dt.days
unit_orders = unit_orders.sort_values(["customer_id", "order_date", "order_id"])
unit_orders["order_number"] = unit_orders.groupby("customer_id").cumcount() + 1

# Convert the marketing source convention YYYY-01-DD into a real month.
unit_marketing = marketing.copy()
unit_marketing["period"] = pd.to_datetime(
    unit_marketing["month"].astype(str).str[:4]
    + "-"
    + unit_marketing["month"].astype(str).str[-2:]
    + "-01"
)

customer_unit = customers.set_index("customer_id")[
    ["acquisition_channel", "first_order_date"]
].copy()
customer_unit["first_order_date"] = pd.to_datetime(customer_unit["first_order_date"])
customer_unit["orders"] = unit_orders.groupby("customer_id")["order_id"].count()
customer_unit["observed_ltv_eur"] = unit_orders.groupby("customer_id")["contribution_eur"].sum()

print("Prepared", len(unit_orders), "orders and", len(customer_unit), "customers.")

### What this preparation tells us

The unit is now a customer whose value is measured in contribution euros. The next step is to establish the **blended baseline**: how much the average acquired customer costs, how much contribution they generate, and when the average customer pays back.

In [ ]:
# Calculate the blended CAC, retention and observed LTV.
paid_spend = unit_marketing["spend_eur"].sum()
blended_cac = paid_spend / len(customer_unit)

customer_unit["retained_90d"] = customer_unit.index.to_series().map(
    unit_orders[unit_orders["days_since_first"].between(1, 90)]
    .groupby("customer_id")["order_id"].nunique()
).fillna(0).gt(0)
customer_unit["retained_180d"] = customer_unit.index.to_series().map(
    unit_orders[unit_orders["days_since_first"].between(1, 180)]
    .groupby("customer_id")["order_id"].nunique()
).fillna(0).gt(0)

blended_summary = pd.Series({
    "customers": len(customer_unit),
    "paid_spend_eur": paid_spend,
    "CAC_eur": blended_cac,
    "retention_90d": customer_unit["retained_90d"].mean(),
    "retention_180d": customer_unit["retained_180d"].mean(),
    "observed_LTV_eur": customer_unit["observed_ltv_eur"].mean(),
    "observed_LTV_CAC": customer_unit["observed_ltv_eur"].mean() / blended_cac,
})
display(blended_summary.round(2))

### Blended result and next question

The blended baseline is **CAC EUR 17.49**, observed contribution LTV **EUR 23.51**, and LTV/CAC **1.34x**. The 90-day retention rate is approximately **18%** and the 180-day retention rate approximately **28%**. Retention is the mechanism that makes the second order important.

This average hides channel mix. The next cell separates paid channels and compares CAC with retention and LTV, so we can identify which acquisition investments are actually self-funding.

In [ ]:
# Compare customer value and acquisition cost by channel.
channel_unit_economics = customer_unit.groupby("acquisition_channel").agg(
    customers=("orders", "size"),
    retention_90d=("retained_90d", "mean"),
    retention_180d=("retained_180d", "mean"),
    observed_ltv_eur=("observed_ltv_eur", "mean"),
    orders_per_customer=("orders", "mean"),
).join(
    unit_marketing.groupby("channel")["spend_eur"].sum().rename("spend_eur")
)
channel_unit_economics["CAC_eur"] = (
    channel_unit_economics["spend_eur"]
    / channel_unit_economics["customers"]
)
channel_unit_economics["observed_LTV_CAC"] = (
    channel_unit_economics["observed_ltv_eur"]
    / channel_unit_economics["CAC_eur"]
)
channel_unit_economics["observed_contribution_after_spend_eur"] = (
    channel_unit_economics["observed_ltv_eur"]
    * channel_unit_economics["customers"]
    - channel_unit_economics["spend_eur"]
)
display(channel_unit_economics.round(2))


Google Shopping is the strongest paid channel: observed LTV/CAC is **1.11x**, with observed payback on order 3. Meta Ads (**0.64x**) and Influencers (**0.59x**) do not recover CAC in observed contribution. Organic/SEO and Email/Referral have no recorded spend, so their CAC is undefined rather than zero.

The next question is whether this is a recent deterioration. We therefore compare acquisition cohorts, using annual spend allocation and equal 90/180-day retention windows.

In [ ]:
# Compare annual acquisition cohorts on equal retention windows.
customer_unit["cohort_year"] = customer_unit["first_order_date"].dt.year.astype(str)
cohort_unit_economics = customer_unit.groupby("cohort_year").agg(
    customers=("orders", "size"),
    retention_90d=("retained_90d", "mean"),
    retention_180d=("retained_180d", "mean"),
    observed_ltv_eur=("observed_ltv_eur", "mean"),
    orders_per_customer=("orders", "mean"),
)
annual_spend = unit_marketing.groupby(unit_marketing["period"].dt.year)["spend_eur"].sum()
cohort_unit_economics["spend_eur"] = [
    annual_spend.get(int(year), np.nan)
    for year in cohort_unit_economics.index
]
cohort_unit_economics["CAC_eur"] = (
    cohort_unit_economics["spend_eur"]
    / cohort_unit_economics["customers"]
)
cohort_unit_economics["observed_LTV_CAC"] = (
    cohort_unit_economics["observed_ltv_eur"]
    / cohort_unit_economics["CAC_eur"]
)
display(cohort_unit_economics.round(2))

### Cohort result and next question

The 2025 cohort has approximately **35% 180-day retention**, **EUR 27.64 observed LTV**, and **1.81x LTV/CAC**. The 2026 cohort falls to approximately **18% retention**, **EUR 17.49 LTV**, and **0.85x LTV/CAC**.

This supports the CFO?s concern, but the result is observational. The final calculation checks payback order directly, which shows whether repeat orders compensate for an unprofitable first order.

In [ ]:
# Find the first order number at which cumulative contribution reaches CAC.
def find_payback_order(customer_ids, cac):
    contribution_by_order = (
        unit_orders[unit_orders["customer_id"].isin(customer_ids)]
        .groupby("order_number")["contribution_eur"]
        .sum()
        .cumsum()
        / len(customer_ids)
    )
    reached = contribution_by_order[contribution_by_order >= cac]
    return reached.index.min() if len(reached) else np.nan

blended_cumulative = (
    unit_orders.groupby("order_number")["contribution_eur"].sum().cumsum()
    / len(customer_unit)
)
blended_payback_order = blended_cumulative[blended_cumulative >= blended_cac].index.min()
channel_unit_economics["payback_order"] = [
    find_payback_order(
        customer_unit.index[customer_unit["acquisition_channel"] == channel],
        channel_unit_economics.loc[channel, "CAC_eur"],
    )
    if pd.notna(channel_unit_economics.loc[channel, "CAC_eur"])
    else np.nan
    for channel in channel_unit_economics.index
]
print("Blended payback order:", blended_payback_order)
display(channel_unit_economics[["CAC_eur", "observed_ltv_eur", "observed_LTV_CAC", "payback_order"]].round(2))

## Unit economics conclusion

- Blended CAC is **EUR 17.49** and blended observed LTV is **EUR 23.51**; contribution reaches CAC on **order 2**.
- Google Shopping is the strongest paid channel, with observed LTV/CAC of **1.11x** and payback on order 3.
- Meta Ads and Influencers do not pay back within the observed order history.
- 2026 cohorts are materially weaker than 2025 cohorts, with lower retention and LTV/CAC below 1x.

**What to do next:** reduce or test Meta/Influencer spend incrementally, protect contribution on the first order, and improve the retention loop through CRM and post-purchase journeys. Treat recent-cohort LTV as a lower bound because recent customers have had less time to repeat.